# JRA-3Q 海面更正気圧（気圧配置・日本域）一括ダウンロード（Colab版）

手元のPC/ネットワークから `github.com` や GDEX（データ配布元）へのHTTPS通信がブロックされる環境向けに、Google Colab上でダウンロードするノートブックです。Colab（Googleのサーバー）から直接ダウンロードするので、手元の回線・セキュリティソフトの制限は関係なくなります。

## 日本域だけに切り出し
GDEXのTHREDDSサーバーが提供する **NetCDF Subset Service (NCSS)** を使い、緯度経度で日本周辺（既定: 北緯15〜50度、東経115〜155度）だけをサーバー側で切り出してもらっています。実測で1ファイル 84MB → 5MB（約94%減）になることを確認済みで、全185ヶ月でも**合計1GB弱**で済みます。

## 保存先について
サイズが小さくなったので、Googleドライブは使わず、Colab上の一時ディスクにまとめてダウンロードしてから、**最後に1回だけZIPにしてブラウザ経由でPCにダウンロード**します（ブラウザの通信は制限されていないので問題なく動くはずです）。

## セッションが切れたら
全体で1GB弱なので、途中で切れても①からやり直すコストはそれほど大きくありません。スクリプトは既にあるファイルをスキップするので、同じセッション内で②を再実行すれば続きから再開されます。セッションごと切れた場合は①からやり直してください（ローカルディスクの中身は消えるので、②から最初のファイルを含め再取得しますが、1ファイルあたり数MBなのですぐ終わります）。

## 使い方
上から順にセルを実行してください（Shift+Enter）。

## ① リポジトリを取得（初回はクローン、2回目以降は最新化のみ）

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/typhoon-dataset-improvements-hn814c'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} https://github.com/awg-yk/typhoon-wind-rainfall {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

## ② ダウンロード実行（本体、日本域に切り出し済み）

地上気圧も欲しい場合は `--include-surface-pressure` を、切り出す範囲を変えたい場合は `--north/--south/--west/--east` を追加してください。

In [ ]:
LOCAL_DIR = '/content/jra3q_pressure'

!cd {REPO_DIR} && python scripts/download_jra3q_pressure.py --out-dir "{LOCAL_DIR}"

## ③ ZIPにまとめてPCへダウンロード

In [ ]:
import shutil
from google.colab import files as colab_files

zip_base = '/content/jra3q_pressure_japan'
zip_path = shutil.make_archive(zip_base, 'zip', LOCAL_DIR)
print(f'{zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
colab_files.download(zip_path)

## ④ （任意）無操作切断を遅らせる

Colabは無操作が続くと自動切断されることがあります。②の実行前にこのセルを流しておくと、ブラウザのタブを開いたままにしている間は接続維持の合図を送り続けます（非公式の小技のため過信せず、切れたら①からやり直してください）。

In [ ]:
from IPython.display import Javascript, display

display(Javascript('''
function KeepAlive(){
  console.log("keep-alive ping");
  document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(KeepAlive, 60000);
'''))

## PCで受け取った後

`jra3q_pressure_japan.zip` を、リポジトリの `data/raw_jra3q/` フォルダなど好きな場所に展開してください。`data/raw_jra3q/` は `.gitignore` 済みなので、そのまま置いてもリポジトリには影響しません。